In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="rnn",
    notebook_name="06_bidirectional_rnns.ipynb"
)

# Bidirectional RNNs -- Reading Forward AND Backward

Up until now, every RNN you have built reads a sequence in one direction: left to right. That is like reading a book for the first time. You pick up clues as you go, but you are always guessing about what comes next.

Now imagine you have already finished the book. You go back and re-read it from the start. This time, because you know the ending, earlier scenes suddenly make way more sense. That is exactly what a **Bidirectional RNN** does -- it reads the sequence from both ends so that every position gets context from the past *and* the future.

We will also look at **stacking** multiple RNN layers on top of each other. Think of it like a team of specialists. The first expert reads the raw data. The second expert reads what the first expert found and summarizes it. The third expert looks at those summaries and makes the big-picture decision. Each layer builds on the previous one's work.

## What You Will Learn

By the end of this notebook, you will understand:
- Why reading only left-to-right is a limitation
- How Bidirectional RNNs read in **both** directions at the same time
- How to **build a Bidirectional RNN from scratch** with NumPy
- How to visualize forward vs backward hidden states
- When to use (and when NOT to use) bidirectional models
- How to **stack** multiple RNN layers for deeper networks

**Prerequisites:** Basic RNN understanding (how hidden states carry information through a sequence).

---

## Jargon Buster

| Term | Plain English Explanation |
|------|---------------------------|
| **Unidirectional RNN** | A normal RNN that reads a sequence in one direction (left to right) |
| **Bidirectional RNN (BiRNN)** | An RNN that reads a sequence in BOTH directions at the same time -- like two readers, one starting from each end |
| **Forward RNN** | The part that reads left to right (the normal direction) |
| **Backward RNN** | The part that reads right to left (reversed) |
| **Concatenation** | Gluing two vectors end-to-end -- if forward gives [1,2] and backward gives [3,4], concatenation gives [1,2,3,4] |
| **Hidden State** | The RNN's "memory" at each time step -- a vector summarizing what it has seen so far |
| **Context** | The surrounding words/tokens that help you understand the meaning of a specific word |
| **Stacked RNN** | Multiple RNN layers stacked on top of each other -- the output of layer 1 feeds into layer 2 |
| **Deep RNN** | Another name for a stacked RNN -- "deep" because it has multiple layers |
| **NER** | Named Entity Recognition -- finding names, places, dates, etc. in text |
| **Sequence Labeling** | Assigning a label to every element in a sequence (e.g., tagging every word in a sentence) |

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

%matplotlib inline

np.random.seed(42)

print("✅ Libraries imported successfully!")
print("📦 NumPy version:", np.__version__)

---

## Part 1: The Limitation of Forward-Only RNNs

### The Mystery Novel Analogy

Imagine you are reading a mystery novel. Halfway through, there is a suspicious scene you cannot quite figure out. Then, near the end, you discover the butler did it. Suddenly that earlier scene makes perfect sense. You needed clues from **later chapters** to understand what happened in **earlier** ones.

A regular (unidirectional) RNN is like a reader who is **forbidden from ever flipping ahead**. It processes each word left-to-right and can only use what it has seen so far. For some tasks that is fine. For others, it is a serious handicap.

### The Classic Example: "Bank"

Look at these two sentences:

1. **"The bank by the river was muddy."** -- bank = river bank (nature)
2. **"The bank sent me a statement."** -- bank = financial institution

When a forward-only RNN reaches the word **"bank"**, it has only seen **"The"** so far. That is not enough context. You need the **rest** of the sentence ("river" or "statement") to know which meaning of "bank" is correct.

A **Bidirectional RNN** solves this by reading from both ends at the same time:
- Forward pass sees: The, bank, by, the, river, was, muddy
- Backward pass sees: muddy, was, river, the, by, bank, The

At the word "bank," the **backward** RNN has already seen "river" -- giving the network the context it needs to pick the right meaning.

In [ ]:
# Demonstrate the ambiguity problem with a simple example

print("🔍 THE AMBIGUITY PROBLEM\n")
print("="*70)

sentence_1 = ["The", "bank", "by", "the", "river", "was", "muddy"]
sentence_2 = ["The", "bank", "sent", "me", "a", "statement"]

print("\nSentence 1:", " ".join(sentence_1))
print("Sentence 2:", " ".join(sentence_2))

print("\n" + "-"*70)
print("\n\u27a1\ufe0f  Forward-only RNN at the word 'bank':")
print("   Has seen:  ['The']")
print("   Hasn't seen: the rest of the sentence")
print("   Problem: Can't tell if it's a RIVER bank or a FINANCIAL bank!")

print("\n\u2194\ufe0f  Bidirectional RNN at the word 'bank':")
print("   Forward has seen:  ['The']")
print("   Backward has seen: ['muddy', 'was', 'river', 'the', 'by']  (Sentence 1)")
print("                  or: ['statement', 'a', 'me', 'sent']         (Sentence 2)")
print("   Solution: The backward RNN provides the missing context!")

print("\n" + "="*70)
print("\n💡 KEY INSIGHT:")
print("   Some words can only be understood by looking at what comes AFTER them.")
print("   Bidirectional RNNs let us peek in both directions!")
print("="*70)

---

## Part 2: Bidirectional RNN Concept

### How It Works

A Bidirectional RNN is surprisingly simple. It runs **two completely separate RNNs** on the same input sequence:

1. **Forward RNN**: Reads the sequence left to right (normal order)
2. **Backward RNN**: Reads the sequence right to left (reversed order)

Picture two runners on the same track, one starting from each end. They both run the full length. When they are done, each runner knows what the entire track looks like -- but from a different perspective. If you combine their observations at every point along the track, you get a much richer picture than either runner could give you alone.

At each time step $t$, we **concatenate** the forward hidden state $\vec{h}_t$ and the backward hidden state $\overleftarrow{h}_t$:

$$h_t = [\vec{h}_t \; ; \; \overleftarrow{h}_t]$$

If each RNN has a hidden size of $H$, then the combined output at each position has size $2H$.

That is really all there is to it. Two RNNs, opposite directions, glue the results together. Let's draw a diagram.

In [ ]:
# Draw a clear diagram of a Bidirectional RNN

fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(-0.5, 7.5)
ax.set_ylim(-0.5, 6.5)
ax.axis('off')

words = ["The", "bank", "by", "the", "river"]
n = len(words)

# Positions
x_positions = [1 + i * 1.4 for i in range(n)]
y_input = 0.5
y_forward = 2.5
y_backward = 4.5
y_output = 6.0

# Draw input words
for i, (x, word) in enumerate(zip(x_positions, words)):
    ax.text(x, y_input, word, ha='center', va='center', fontsize=13,
            fontweight='bold', bbox=dict(boxstyle='round,pad=0.3',
            facecolor='lightyellow', edgecolor='orange', linewidth=2))

# Draw forward RNN cells (blue)
for i, x in enumerate(x_positions):
    circle = plt.Circle((x, y_forward), 0.35, color='#AED6F1', ec='#2980B9', linewidth=2, zorder=3)
    ax.add_patch(circle)
    ax.text(x, y_forward, f'$\\vec{{h}}_{i}$', ha='center', va='center', fontsize=11)
    # Arrow from input to forward cell
    ax.annotate('', xy=(x, y_forward - 0.35), xytext=(x, y_input + 0.25),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='gray'))

# Forward arrows between cells (left to right)
for i in range(n - 1):
    ax.annotate('', xy=(x_positions[i+1] - 0.35, y_forward),
                xytext=(x_positions[i] + 0.35, y_forward),
                arrowprops=dict(arrowstyle='->', lw=2.5, color='#2980B9'))

# Draw backward RNN cells (red/orange)
for i, x in enumerate(x_positions):
    circle = plt.Circle((x, y_backward), 0.35, color='#F5B7B1', ec='#E74C3C', linewidth=2, zorder=3)
    ax.add_patch(circle)
    ax.text(x, y_backward, f'$\\overleftarrow{{h}}_{i}$', ha='center', va='center', fontsize=11)
    # Arrow from input to backward cell
    ax.annotate('', xy=(x, y_backward - 0.35), xytext=(x, y_input + 0.25),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='gray'))

# Backward arrows between cells (right to left)
for i in range(n - 1, 0, -1):
    ax.annotate('', xy=(x_positions[i-1] + 0.35, y_backward),
                xytext=(x_positions[i] - 0.35, y_backward),
                arrowprops=dict(arrowstyle='->', lw=2.5, color='#E74C3C'))

# Draw concatenated output
for i, x in enumerate(x_positions):
    rect = plt.Rectangle((x - 0.45, y_output - 0.2), 0.9, 0.4,
                          facecolor='#D5F5E3', edgecolor='#27AE60', linewidth=2, zorder=3)
    ax.add_patch(rect)
    ax.text(x, y_output, f'[$\\vec{{h}}_{i}$;$\\overleftarrow{{h}}_{i}$]',
            ha='center', va='center', fontsize=9)
    # Arrows from forward and backward to output
    ax.annotate('', xy=(x - 0.15, y_output - 0.2), xytext=(x - 0.15, y_forward + 0.35),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='#2980B9'))
    ax.annotate('', xy=(x + 0.15, y_output - 0.2), xytext=(x + 0.15, y_backward + 0.35),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='#E74C3C'))

# Labels
ax.text(-0.3, y_input, 'Input', ha='right', va='center', fontsize=12, fontweight='bold', color='orange')
ax.text(-0.3, y_forward, 'Forward\nRNN', ha='right', va='center', fontsize=12,
         fontweight='bold', color='#2980B9')
ax.text(-0.3, y_backward, 'Backward\nRNN', ha='right', va='center', fontsize=12,
         fontweight='bold', color='#E74C3C')
ax.text(-0.3, y_output, 'Combined\nOutput', ha='right', va='center', fontsize=12,
         fontweight='bold', color='#27AE60')

# Direction labels
ax.text(x_positions[2], y_forward - 0.7, '\u27a1\ufe0f Forward (left to right)',
         ha='center', fontsize=11, color='#2980B9', fontweight='bold')
ax.text(x_positions[2], y_backward + 0.7, '\u2b05\ufe0f Backward (right to left)',
         ha='center', fontsize=11, color='#E74C3C', fontweight='bold')

ax.set_title('Bidirectional RNN Architecture', fontsize=16, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

print("\n📊 What you\u2019re seeing:")
print("   \u2022 Yellow boxes = input words")
print("   \u2022 Blue circles = forward RNN hidden states (reads left \u2192 right)")
print("   \u2022 Red circles  = backward RNN hidden states (reads right \u2192 left)")
print("   \u2022 Green boxes  = concatenated output (both directions combined)")
print("\n   At each position, we get context from BOTH sides of the sequence!")

---

## Part 3: Build a Bidirectional RNN from Scratch

Time to build one ourselves. We will create:
1. A basic **RNN cell** (the building block)
2. A **Bidirectional RNN** that runs two cells in opposite directions

### The Math (Quick Recap)

A single RNN cell at time step $t$:

$$h_t = \tanh(W_{xh} \cdot x_t + W_{hh} \cdot h_{t-1} + b_h)$$

For a Bidirectional RNN we simply run **two** of these -- one on the original sequence, one on the reversed sequence -- then concatenate their hidden states.

In [ ]:
class RNNCell:
    """
    A single vanilla RNN cell.
    
    At each time step it takes an input x_t and the previous hidden state h_{t-1},
    and produces the next hidden state h_t.
    
    h_t = tanh(W_xh @ x_t + W_hh @ h_{t-1} + b_h)
    """
    
    def __init__(self, input_size, hidden_size):
        """
        Parameters:
        - input_size:  dimension of each input vector x_t
        - hidden_size: dimension of the hidden state h_t
        """
        self.input_size = input_size
        self.hidden_size = hidden_size
        
        # Xavier initialization for stable training
        scale_xh = np.sqrt(2.0 / (input_size + hidden_size))
        scale_hh = np.sqrt(2.0 / (hidden_size + hidden_size))
        
        self.W_xh = np.random.randn(hidden_size, input_size) * scale_xh   # input -> hidden
        self.W_hh = np.random.randn(hidden_size, hidden_size) * scale_hh  # hidden -> hidden
        self.b_h  = np.zeros((hidden_size, 1))                            # bias
    
    def forward_step(self, x_t, h_prev):
        """
        One time step of the RNN.
        
        Parameters:
        - x_t:    input at time t, shape (input_size, 1)
        - h_prev: hidden state from time t-1, shape (hidden_size, 1)
        
        Returns:
        - h_t: new hidden state, shape (hidden_size, 1)
        """
        h_t = np.tanh(self.W_xh @ x_t + self.W_hh @ h_prev + self.b_h)
        return h_t
    
    def forward_sequence(self, X):
        """
        Run the RNN over an entire sequence.
        
        Parameters:
        - X: input sequence, shape (seq_len, input_size)
        
        Returns:
        - hidden_states: array of shape (seq_len, hidden_size)
        """
        seq_len = X.shape[0]
        h = np.zeros((self.hidden_size, 1))  # initial hidden state
        hidden_states = []
        
        for t in range(seq_len):
            x_t = X[t].reshape(-1, 1)  # column vector
            h = self.forward_step(x_t, h)
            hidden_states.append(h.flatten())
        
        return np.array(hidden_states)  # (seq_len, hidden_size)


print("✅ RNNCell class defined!")
print("   This is the building block for our Bidirectional RNN.")

In [ ]:
class BidirectionalRNN:
    """
    Bidirectional RNN: runs two RNN cells in opposite directions and
    concatenates their hidden states.
    
    Forward RNN:  reads x_0, x_1, ..., x_T   (left to right)
    Backward RNN: reads x_T, x_{T-1}, ..., x_0  (right to left)
    
    Output at time t = [forward_h_t ; backward_h_t]  (size = 2 * hidden_size)
    """
    
    def __init__(self, input_size, hidden_size):
        """
        Parameters:
        - input_size:  dimension of each input vector
        - hidden_size: hidden size for EACH direction
                       (total output size will be 2 * hidden_size)
        """
        self.hidden_size = hidden_size
        
        # Two completely separate RNN cells
        self.forward_rnn  = RNNCell(input_size, hidden_size)
        self.backward_rnn = RNNCell(input_size, hidden_size)
    
    def forward(self, X):
        """
        Run the bidirectional RNN on input sequence X.
        
        Parameters:
        - X: input sequence, shape (seq_len, input_size)
        
        Returns:
        - outputs:         concatenated hidden states, shape (seq_len, 2*hidden_size)
        - forward_states:  forward hidden states, shape (seq_len, hidden_size)
        - backward_states: backward hidden states, shape (seq_len, hidden_size)
        """
        seq_len = X.shape[0]
        
        # 1. Forward RNN: read left to right
        forward_states = self.forward_rnn.forward_sequence(X)
        
        # 2. Backward RNN: read right to left (reverse the input!)
        X_reversed = X[::-1]  # reverse the sequence
        backward_states_reversed = self.backward_rnn.forward_sequence(X_reversed)
        
        # IMPORTANT: reverse the backward states so they align with the original positions
        # backward_states_reversed[0] corresponds to the LAST input word
        # We flip it so backward_states[t] aligns with input position t
        backward_states = backward_states_reversed[::-1]
        
        # 3. Concatenate at each position
        outputs = np.concatenate([forward_states, backward_states], axis=1)
        
        return outputs, forward_states, backward_states


print("✅ BidirectionalRNN class defined!")
print("\n📋 How it works:")
print("   1. Forward RNN reads the sequence normally (left to right)")
print("   2. Backward RNN reads the REVERSED sequence (right to left)")
print("   3. We reverse the backward outputs to re-align with original positions")
print("   4. Concatenate forward and backward hidden states at each position")
print("   5. Output size = 2 \u00d7 hidden_size")

In [ ]:
# Let's test our Bidirectional RNN!

print("🧪 TESTING THE BIDIRECTIONAL RNN\n")
print("="*70)

# Create a simple sequence: 5 time steps, 3-dimensional input
# Think of this as a sentence with 5 words, each represented by a 3D vector
np.random.seed(42)

seq_len = 5
input_size = 3
hidden_size = 4

# Random input sequence
X = np.random.randn(seq_len, input_size)

print(f"Input sequence shape: {X.shape}  (seq_len={seq_len}, input_size={input_size})")
print(f"Hidden size per direction: {hidden_size}")
print(f"Expected output size: {2 * hidden_size} (2 \u00d7 {hidden_size})")

# Create and run the BiRNN
birnn = BidirectionalRNN(input_size, hidden_size)
outputs, forward_states, backward_states = birnn.forward(X)

print(f"\n📊 Results:")
print(f"   Forward states shape:  {forward_states.shape}  (seq_len, hidden_size)")
print(f"   Backward states shape: {backward_states.shape}  (seq_len, hidden_size)")
print(f"   Combined output shape: {outputs.shape}  (seq_len, 2\u00d7hidden_size)")

print(f"\n🔍 Verifying concatenation at position 0:")
print(f"   Forward  h_0: {forward_states[0][:3]}...")
print(f"   Backward h_0: {backward_states[0][:3]}...")
print(f"   Combined[0]:  {outputs[0][:3]}... {outputs[0][4:7]}...")
print(f"   Match: {np.allclose(outputs[0], np.concatenate([forward_states[0], backward_states[0]]))}")

print("\n" + "="*70)
print("\n💡 KEY POINT:")
print("   Each position now has a representation from BOTH directions.")
print(f"   The output is {2 * hidden_size}D instead of {hidden_size}D \u2014 richer context!")
print("="*70)

---

## Part 4: Visualize Forward vs Backward Hidden States

Let's see what each direction actually captures. We will feed a short sentence through our BiRNN and plot the hidden state activations at every position.

In [ ]:
# Create a more interpretable example with word embeddings

np.random.seed(123)

# Simulate a sentence with simple "word embeddings"
words = ["The", "bank", "by", "the", "river", "was", "muddy"]
vocab = {w: i for i, w in enumerate(sorted(set(words)))}

# Create simple one-hot-ish embeddings (7 words, 8-dim embedding)
embedding_dim = 8
embeddings = {}
for w in vocab:
    np.random.seed(hash(w) % 2**31)
    embeddings[w] = np.random.randn(embedding_dim) * 0.5

# Build input sequence
X_sentence = np.array([embeddings[w] for w in words])

print(f"Sentence: {' '.join(words)}")
print(f"Input shape: {X_sentence.shape} (seq_len={len(words)}, embedding_dim={embedding_dim})")

# Run through BiRNN
hidden_size_viz = 6
birnn_viz = BidirectionalRNN(embedding_dim, hidden_size_viz)
outputs_viz, fwd_states, bwd_states = birnn_viz.forward(X_sentence)

print(f"\nForward states:  {fwd_states.shape}")
print(f"Backward states: {bwd_states.shape}")
print(f"Combined output: {outputs_viz.shape}")

In [ ]:
# Visualize the hidden states

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Forward hidden states
im1 = axes[0].imshow(fwd_states.T, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
axes[0].set_xticks(range(len(words)))
axes[0].set_xticklabels(words, rotation=45, ha='right', fontsize=11)
axes[0].set_ylabel('Hidden Unit', fontsize=12, fontweight='bold')
axes[0].set_title('\u27a1\ufe0f Forward Hidden States\n(left to right)', fontsize=13, fontweight='bold', color='#2980B9')
axes[0].set_yticks(range(hidden_size_viz))
plt.colorbar(im1, ax=axes[0], shrink=0.8)

# Plot 2: Backward hidden states
im2 = axes[1].imshow(bwd_states.T, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
axes[1].set_xticks(range(len(words)))
axes[1].set_xticklabels(words, rotation=45, ha='right', fontsize=11)
axes[1].set_ylabel('Hidden Unit', fontsize=12, fontweight='bold')
axes[1].set_title('\u2b05\ufe0f Backward Hidden States\n(right to left)', fontsize=13, fontweight='bold', color='#E74C3C')
axes[1].set_yticks(range(hidden_size_viz))
plt.colorbar(im2, ax=axes[1], shrink=0.8)

# Plot 3: Combined (concatenated)
im3 = axes[2].imshow(outputs_viz.T, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
axes[2].set_xticks(range(len(words)))
axes[2].set_xticklabels(words, rotation=45, ha='right', fontsize=11)
axes[2].set_ylabel('Hidden Unit', fontsize=12, fontweight='bold')
axes[2].set_title('\u2194\ufe0f Combined (Concatenated)\nFull bidirectional context', fontsize=13, fontweight='bold', color='#27AE60')
axes[2].set_yticks(range(2 * hidden_size_viz))
# Add a horizontal line to show where forward ends and backward begins
axes[2].axhline(y=hidden_size_viz - 0.5, color='black', linestyle='--', linewidth=2)
axes[2].text(len(words) + 0.6, hidden_size_viz / 2 - 0.5, 'Fwd', fontsize=10, va='center', color='#2980B9', fontweight='bold')
axes[2].text(len(words) + 0.6, hidden_size_viz + hidden_size_viz / 2 - 0.5, 'Bwd', fontsize=10, va='center', color='#E74C3C', fontweight='bold')
plt.colorbar(im3, ax=axes[2], shrink=0.8)

plt.suptitle('"The bank by the river was muddy"', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n💡 What to notice:")
print('   \u2022 Forward states: at "bank" (position 1), only "The" has been seen')
print('   \u2022 Backward states: at "bank" (position 1), "river", "was", "muddy" have been seen')
print('   \u2022 Combined: at "bank" we have context from BOTH sides!')
print('   \u2022 The combined representation is 2\u00d7 taller (more dimensions = richer)')

In [ ]:
# Show what information each direction captures at the word "bank"

bank_idx = 1  # "bank" is at index 1

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Forward hidden state at "bank"
axes[0].bar(range(hidden_size_viz), fwd_states[bank_idx], color='#AED6F1', edgecolor='#2980B9', linewidth=2)
axes[0].set_title(f'Forward h at "{words[bank_idx]}"\nHas seen: {words[:bank_idx+1]}', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Hidden Unit', fontsize=11)
axes[0].set_ylabel('Activation', fontsize=11)
axes[0].set_ylim(-1, 1)
axes[0].grid(axis='y', alpha=0.3)

# Backward hidden state at "bank"
axes[1].bar(range(hidden_size_viz), bwd_states[bank_idx], color='#F5B7B1', edgecolor='#E74C3C', linewidth=2)
axes[1].set_title(f'Backward h at "{words[bank_idx]}"\nHas seen: {words[bank_idx:][::-1]}', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Hidden Unit', fontsize=11)
axes[1].set_ylabel('Activation', fontsize=11)
axes[1].set_ylim(-1, 1)
axes[1].grid(axis='y', alpha=0.3)

# Combined
combined = outputs_viz[bank_idx]
colors = ['#AED6F1'] * hidden_size_viz + ['#F5B7B1'] * hidden_size_viz
edgecolors = ['#2980B9'] * hidden_size_viz + ['#E74C3C'] * hidden_size_viz
axes[2].bar(range(2 * hidden_size_viz), combined, color=colors, edgecolor=edgecolors, linewidth=2)
axes[2].axvline(x=hidden_size_viz - 0.5, color='black', linestyle='--', linewidth=2)
axes[2].set_title(f'Combined at "{words[bank_idx]}"\nContext from BOTH sides', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Hidden Unit (blue=fwd, red=bwd)', fontsize=11)
axes[2].set_ylabel('Activation', fontsize=11)
axes[2].set_ylim(-1, 1)
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print('\n💡 At the word "bank":') 
print('   \u2022 Forward RNN has only seen ["The", "bank"] \u2014 not enough context!')
print('   \u2022 Backward RNN has seen ["muddy", "was", "river", "the", "by", "bank"] \u2014 knows about "river"!')
print('   \u2022 Combined: the network has ALL the context it needs to disambiguate "bank"')

---

## Part 5: When to Use Bidirectional RNNs

Bidirectional RNNs are powerful, but they are **not always the right choice**. The deciding question is simple:

> **Do you have the FULL sequence available before you need to make a prediction?**

If yes, go bidirectional. If no, stick with unidirectional. That is the entire rule.

### Good Use Cases (Full sequence available)

| Task | Why BiRNN helps |
|------|------------------|
| **Text Classification** | You have the entire review/document before deciding "positive" or "negative" |
| **Named Entity Recognition (NER)** | You need surrounding context to decide if "Apple" is a company or a fruit |
| **Machine Translation (encoder)** | The encoder reads the full source sentence before translation begins |
| **Sentiment Analysis** | "The movie was not bad at all" -- you need to see "at all" to understand "not bad" |
| **Question Answering** | You read the full passage before answering |

### Bad Use Cases (You cannot see the future)

| Task | Why BiRNN does NOT work |
|------|-------------------------|
| **Text Generation** | You predict the NEXT word -- you cannot peek at words that do not exist yet |
| **Real-time Speech Recognition** | Audio arrives one frame at a time -- you cannot wait for the full utterance |
| **Live Stock Prediction** | You only have data up to "now" -- future data has not happened yet |
| **Chatbot Response** | You generate token by token; future tokens are unknown |

### Decision Rule

- **Full input available** before prediction --> use Bidirectional
- **Generating output token by token** --> use Unidirectional
- If you are unsure, ask yourself: "Can I wait until the entire input is collected before I produce any output?" If yes, bidirectional. If no, unidirectional.

In [ ]:
# Visualize the decision: When to use BiRNN?

fig, ax = plt.subplots(figsize=(12, 6))
ax.axis('off')

# Title
ax.text(0.5, 0.95, 'When to Use Bidirectional RNNs?', ha='center', va='center',
         fontsize=18, fontweight='bold', transform=ax.transAxes)

# Decision box
decision_box = mpatches.FancyBboxPatch((0.25, 0.7), 0.5, 0.12, boxstyle='round,pad=0.02',
                                        facecolor='#F9E79F', edgecolor='#F39C12', linewidth=3,
                                        transform=ax.transAxes)
ax.add_patch(decision_box)
ax.text(0.5, 0.76, 'Do you have the FULL sequence\nbefore making predictions?',
         ha='center', va='center', fontsize=13, fontweight='bold', transform=ax.transAxes)

# YES branch
ax.annotate('YES', xy=(0.35, 0.7), xytext=(0.2, 0.55),
             fontsize=14, fontweight='bold', color='#27AE60', ha='center',
             arrowprops=dict(arrowstyle='->', lw=3, color='#27AE60'),
             transform=ax.transAxes)

yes_box = mpatches.FancyBboxPatch((0.02, 0.3), 0.4, 0.2, boxstyle='round,pad=0.02',
                                   facecolor='#D5F5E3', edgecolor='#27AE60', linewidth=3,
                                   transform=ax.transAxes)
ax.add_patch(yes_box)
ax.text(0.22, 0.43, '\u2705 Use Bidirectional!', ha='center', va='center',
         fontsize=13, fontweight='bold', color='#27AE60', transform=ax.transAxes)
ax.text(0.22, 0.35, '\u2022 Text classification\n\u2022 Named Entity Recognition\n\u2022 Sentiment analysis\n\u2022 Machine translation (encoder)',
         ha='center', va='center', fontsize=10, transform=ax.transAxes)

# NO branch
ax.annotate('NO', xy=(0.65, 0.7), xytext=(0.8, 0.55),
             fontsize=14, fontweight='bold', color='#E74C3C', ha='center',
             arrowprops=dict(arrowstyle='->', lw=3, color='#E74C3C'),
             transform=ax.transAxes)

no_box = mpatches.FancyBboxPatch((0.58, 0.3), 0.4, 0.2, boxstyle='round,pad=0.02',
                                  facecolor='#FADBD8', edgecolor='#E74C3C', linewidth=3,
                                  transform=ax.transAxes)
ax.add_patch(no_box)
ax.text(0.78, 0.43, '\u274c Use Unidirectional!', ha='center', va='center',
         fontsize=13, fontweight='bold', color='#E74C3C', transform=ax.transAxes)
ax.text(0.78, 0.35, '\u2022 Text generation\n\u2022 Real-time streaming\n\u2022 Chatbot responses\n\u2022 Live predictions',
         ha='center', va='center', fontsize=10, transform=ax.transAxes)

# Bottom note
ax.text(0.5, 0.08, '💡 Bidirectional = better context, but requires the full input upfront',
         ha='center', va='center', fontsize=12, fontstyle='italic',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8),
         transform=ax.transAxes)

plt.tight_layout()
plt.show()

---

## Part 6: Stacked (Deep) RNNs

### Why Stack Layers?

A single-layer RNN can learn patterns, but **stacking** multiple layers lets the network learn increasingly **abstract** features. Think of it like a team of specialists working on the same project:

- **Person 1** (Layer 1) reads the raw data and picks out the basic patterns -- individual word meanings, simple relationships.
- **Person 2** (Layer 2) reads what Person 1 found and summarizes it into higher-level insights -- phrases, grammar, relationships between ideas.
- **Person 3** (Layer 3) looks at those summaries and makes the big-picture decision -- the overall sentiment, the topic, the intent.

Each person is an expert at their level of abstraction. None of them could do the whole job alone, but together they handle tasks that are too complex for a single layer.

### How It Works

1. Layer 1's **output** becomes Layer 2's **input**
2. Layer 2's output becomes Layer 3's input
3. And so on...

You can also make **each** layer bidirectional. That gives you the best of both worlds: multiple levels of abstraction, each with full left-and-right context.

In [ ]:
class StackedBidirectionalRNN:
    """
    A multi-layer (stacked) Bidirectional RNN.
    
    Each layer is a full BiRNN. The output of one layer (2*hidden_size)
    becomes the input of the next layer.
    """
    
    def __init__(self, input_size, hidden_size, num_layers):
        """
        Parameters:
        - input_size:  dimension of the original input
        - hidden_size: hidden size per direction per layer
        - num_layers:  number of stacked BiRNN layers
        """
        self.num_layers = num_layers
        self.hidden_size = hidden_size
        self.layers = []
        
        for i in range(num_layers):
            # First layer takes original input_size
            # Subsequent layers take 2*hidden_size (output of previous BiRNN)
            layer_input_size = input_size if i == 0 else 2 * hidden_size
            self.layers.append(BidirectionalRNN(layer_input_size, hidden_size))
    
    def forward(self, X):
        """
        Run input through all stacked BiRNN layers.
        
        Returns:
        - final_output:   output of the last layer, shape (seq_len, 2*hidden_size)
        - all_layer_outputs: list of outputs from each layer (for visualization)
        """
        current_input = X
        all_layer_outputs = []
        
        for i, layer in enumerate(self.layers):
            output, fwd, bwd = layer.forward(current_input)
            all_layer_outputs.append({'output': output, 'forward': fwd, 'backward': bwd})
            current_input = output  # feed into next layer
        
        return current_input, all_layer_outputs


# Test it!
print("🧪 TESTING STACKED BIDIRECTIONAL RNN\n")
print("="*70)

np.random.seed(42)
input_size = 3
hidden_size = 4
num_layers = 3
seq_len = 5

X_test = np.random.randn(seq_len, input_size)

stacked_birnn = StackedBidirectionalRNN(input_size, hidden_size, num_layers)
final_out, all_outputs = stacked_birnn.forward(X_test)

print(f"Input shape:  {X_test.shape}  (seq_len={seq_len}, input_size={input_size})")
print(f"Number of layers: {num_layers}")
print(f"Hidden size per direction: {hidden_size}")
print()

for i, layer_out in enumerate(all_outputs):
    print(f"Layer {i+1} output shape: {layer_out['output'].shape}")

print(f"\nFinal output shape: {final_out.shape}")
print(f"\n💡 Each layer\'s output (size {2*hidden_size}) feeds into the next layer.")
print(f"   Layer 1 input size: {input_size} (original)")
for i in range(1, num_layers):
    print(f"   Layer {i+1} input size: {2*hidden_size} (output of layer {i})")
print("="*70)

In [ ]:
# Draw a diagram of a 2-layer Bidirectional RNN

fig, ax = plt.subplots(figsize=(14, 9))
ax.set_xlim(-1, 8)
ax.set_ylim(-0.5, 10.5)
ax.axis('off')

words_diag = ["x_0", "x_1", "x_2", "x_3"]
n_diag = len(words_diag)
x_pos = [1.0 + i * 1.8 for i in range(n_diag)]

y_input = 0.5

# Layer 1
y_fwd1 = 2.2
y_bwd1 = 3.8
y_concat1 = 5.0

# Layer 2
y_fwd2 = 6.2
y_bwd2 = 7.8
y_concat2 = 9.0

def draw_layer(ax, x_positions, y_fwd, y_bwd, y_concat, layer_num, n):
    """Draw one bidirectional layer."""
    # Forward cells
    for i, x in enumerate(x_positions):
        circle = plt.Circle((x, y_fwd), 0.3, color='#AED6F1', ec='#2980B9', linewidth=2, zorder=3)
        ax.add_patch(circle)
        ax.text(x, y_fwd, f'$\\vec{{h}}^{layer_num}_{i}$', ha='center', va='center', fontsize=9)
    # Forward arrows
    for i in range(n - 1):
        ax.annotate('', xy=(x_positions[i+1] - 0.3, y_fwd),
                    xytext=(x_positions[i] + 0.3, y_fwd),
                    arrowprops=dict(arrowstyle='->', lw=2, color='#2980B9'))
    
    # Backward cells
    for i, x in enumerate(x_positions):
        circle = plt.Circle((x, y_bwd), 0.3, color='#F5B7B1', ec='#E74C3C', linewidth=2, zorder=3)
        ax.add_patch(circle)
        ax.text(x, y_bwd, f'$\\overleftarrow{{h}}^{layer_num}_{i}$', ha='center', va='center', fontsize=9)
    # Backward arrows
    for i in range(n - 1, 0, -1):
        ax.annotate('', xy=(x_positions[i-1] + 0.3, y_bwd),
                    xytext=(x_positions[i] - 0.3, y_bwd),
                    arrowprops=dict(arrowstyle='->', lw=2, color='#E74C3C'))
    
    # Concatenated output
    for i, x in enumerate(x_positions):
        rect = plt.Rectangle((x - 0.4, y_concat - 0.18), 0.8, 0.36,
                              facecolor='#D5F5E3', edgecolor='#27AE60', linewidth=2, zorder=3)
        ax.add_patch(rect)
        ax.text(x, y_concat, f'L{layer_num} out', ha='center', va='center', fontsize=8, fontweight='bold')
        # Arrows from fwd and bwd to concat
        ax.annotate('', xy=(x - 0.1, y_concat - 0.18), xytext=(x - 0.1, y_fwd + 0.3),
                    arrowprops=dict(arrowstyle='->', lw=1, color='#2980B9', alpha=0.6))
        ax.annotate('', xy=(x + 0.1, y_concat - 0.18), xytext=(x + 0.1, y_bwd + 0.3),
                    arrowprops=dict(arrowstyle='->', lw=1, color='#E74C3C', alpha=0.6))

# Draw input
for i, (x, w) in enumerate(zip(x_pos, words_diag)):
    ax.text(x, y_input, w, ha='center', va='center', fontsize=13, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor='orange', linewidth=2))

# Arrows from input to Layer 1 cells
for x in x_pos:
    ax.annotate('', xy=(x, y_fwd1 - 0.3), xytext=(x, y_input + 0.25),
                arrowprops=dict(arrowstyle='->', lw=1.2, color='gray'))
    ax.annotate('', xy=(x, y_bwd1 - 0.3), xytext=(x, y_input + 0.25),
                arrowprops=dict(arrowstyle='->', lw=1.2, color='gray'))

# Draw Layer 1
draw_layer(ax, x_pos, y_fwd1, y_bwd1, y_concat1, 1, n_diag)

# Arrows from Layer 1 output to Layer 2 cells
for x in x_pos:
    ax.annotate('', xy=(x, y_fwd2 - 0.3), xytext=(x, y_concat1 + 0.18),
                arrowprops=dict(arrowstyle='->', lw=1.2, color='gray'))
    ax.annotate('', xy=(x, y_bwd2 - 0.3), xytext=(x, y_concat1 + 0.18),
                arrowprops=dict(arrowstyle='->', lw=1.2, color='gray'))

# Draw Layer 2
draw_layer(ax, x_pos, y_fwd2, y_bwd2, y_concat2, 2, n_diag)

# Labels on the left
ax.text(-0.8, y_input, 'Input', ha='center', va='center', fontsize=12, fontweight='bold', color='orange')
ax.text(-0.8, (y_fwd1 + y_bwd1) / 2, 'Layer 1\n(BiRNN)', ha='center', va='center',
         fontsize=12, fontweight='bold', color='gray')
ax.text(-0.8, y_concat1, 'Concat', ha='center', va='center', fontsize=10, fontweight='bold', color='#27AE60')
ax.text(-0.8, (y_fwd2 + y_bwd2) / 2, 'Layer 2\n(BiRNN)', ha='center', va='center',
         fontsize=12, fontweight='bold', color='gray')
ax.text(-0.8, y_concat2, 'Final\nOutput', ha='center', va='center', fontsize=10, fontweight='bold', color='#27AE60')

ax.set_title('2-Layer Stacked Bidirectional RNN', fontsize=16, fontweight='bold', pad=10)

# Legend
legend_elements = [
    mpatches.Patch(facecolor='#AED6F1', edgecolor='#2980B9', label='Forward RNN'),
    mpatches.Patch(facecolor='#F5B7B1', edgecolor='#E74C3C', label='Backward RNN'),
    mpatches.Patch(facecolor='#D5F5E3', edgecolor='#27AE60', label='Concatenated Output'),
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=11)

plt.tight_layout()
plt.show()

print("\n📋 Architecture summary:")
print("   Layer 1: BiRNN reads original input")
print("            Output size = 2 \u00d7 hidden_size")
print("   Layer 2: BiRNN reads Layer 1\u2019s concatenated output")
print("            Output size = 2 \u00d7 hidden_size")
print("\n   Layer 1 captures low-level patterns")
print("   Layer 2 captures higher-level, more abstract patterns")

---

## Bonus: The Disambiguation Challenge

Let's make this concrete with an actual disambiguation task. We'll create pairs of sentences where one ambiguous word means different things depending on context that appears AFTER it.

A forward-only RNN at the ambiguous word has no idea which meaning is correct. A bidirectional RNN can use the backward pass to peek at future words and resolve the ambiguity.

We'll measure the **disambiguation score**: how different are the hidden state representations for the two meanings at the ambiguous word? Bigger difference = easier to tell them apart.

In [ ]:
# THE DISAMBIGUATION CHALLENGE: Forward vs Bidirectional RNN

print("THE DISAMBIGUATION CHALLENGE")
print("=" * 65)
print("Can a bidirectional RNN distinguish ambiguous words better")
print("than a forward-only RNN? Let's find out.")
print("=" * 65)

np.random.seed(42)

# Sentence pairs where the SAME first few words lead to different meanings
# The disambiguating context comes AFTER the ambiguous word
disambiguation_pairs = [
    {
        "word": "bank",
        "sent_A": "The bank by the river was flooded last week",
        "sent_B": "The bank sent me a statement about my account",
        "disambig_pos": 1,  # "bank" is at position 1
    },
    {
        "word": "bat",
        "sent_A": "The bat flew out of the dark cave at dusk",
        "sent_B": "The bat cracked as he hit the baseball hard",
        "disambig_pos": 1,
    },
    {
        "word": "cold",
        "sent_A": "The cold winter wind blew through the streets",
        "sent_B": "She caught a cold after being out in the rain",
        "disambig_pos": 1,
    },
]

def encode_sentence_chars(sentence, max_chars=50):
    """One-hot encode a sentence character by character."""
    all_chars = "abcdefghijklmnopqrstuvwxyz "
    char_to_idx = {ch: i for i, ch in enumerate(all_chars)}
    vocab_size = len(all_chars)
    
    encoded = []
    for ch in sentence.lower()[:max_chars]:
        if ch in char_to_idx:
            vec = np.zeros(vocab_size)
            vec[char_to_idx[ch]] = 1.0
        else:
            vec = np.zeros(vocab_size)
        encoded.append(vec)
    return np.array(encoded)

input_size_dis = 27  # 26 letters + space
hidden_size_dis = 16

# Create models
np.random.seed(42)
forward_rnn = RNNCell(input_size_dis, hidden_size_dis)
birnn_dis = BidirectionalRNN(input_size_dis, hidden_size_dis)

results = []

print(f"\n{'Word':>6} | {'Forward RNN':>14} | {'BiRNN':>12} | {'Winner':>10}")
print("-" * 50)

for pair in disambiguation_pairs:
    word = pair["word"]
    
    # Encode both sentences
    X_A = encode_sentence_chars(pair["sent_A"])
    X_B = encode_sentence_chars(pair["sent_B"])
    
    # Find the character position of the ambiguous word
    # (it's always near the start -- word position 1 = after "The ")
    disambig_char_pos = 4  # "The " = 4 chars, then the word starts
    
    # Forward RNN: hidden state at the FIRST CHARACTER of the ambiguous word
    fwd_states_A = forward_rnn.forward_sequence(X_A)
    fwd_states_B = forward_rnn.forward_sequence(X_B)
    
    fwd_h_A = fwd_states_A[disambig_char_pos]
    fwd_h_B = fwd_states_B[disambig_char_pos]
    fwd_distance = np.linalg.norm(fwd_h_A - fwd_h_B)
    
    # Bidirectional RNN: concatenated hidden state at the ambiguous word position
    bi_out_A, bi_fwd_A, bi_bwd_A = birnn_dis.forward(X_A)
    bi_out_B, bi_fwd_B, bi_bwd_B = birnn_dis.forward(X_B)
    
    bi_h_A = bi_out_A[disambig_char_pos]
    bi_h_B = bi_out_B[disambig_char_pos]
    bi_distance = np.linalg.norm(bi_h_A - bi_h_B)
    
    winner = "BiRNN" if bi_distance > fwd_distance else "Forward"
    improvement = (bi_distance / max(fwd_distance, 1e-6) - 1) * 100
    
    results.append({
        "word": word, "fwd": fwd_distance, "bi": bi_distance,
        "winner": winner, "improvement": improvement
    })
    
    print(f"{word:>6} | {fwd_distance:>14.4f} | {bi_distance:>12.4f} | {winner:>10} (+{improvement:.0f}%)")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

words_list = [r["word"] for r in results]
fwd_scores = [r["fwd"] for r in results]
bi_scores = [r["bi"] for r in results]

x = np.arange(len(words_list))
width = 0.35

bars1 = axes[0].bar(x - width/2, fwd_scores, width, label='Forward RNN',
                    color='#E74C3C', alpha=0.8, edgecolor='black')
bars2 = axes[0].bar(x + width/2, bi_scores, width, label='Bidirectional RNN',
                    color='#27AE60', alpha=0.8, edgecolor='black')

axes[0].set_xticks(x)
axes[0].set_xticklabels([f'"{w}"' for w in words_list], fontsize=12, fontweight='bold')
axes[0].set_ylabel('Hidden State Distance Between Meanings', fontsize=11)
axes[0].set_title('Disambiguation Score at Ambiguous Word\n(Higher = easier to tell the two meanings apart)',
                  fontsize=12, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(axis='y', alpha=0.3)

# Add value labels
for bar in bars1:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{bar.get_height():.3f}', ha='center', fontsize=9)
for bar in bars2:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{bar.get_height():.3f}', ha='center', fontsize=9)

# Improvement chart
improvements = [r["improvement"] for r in results]
colors_imp = ['#27AE60' if imp > 0 else '#E74C3C' for imp in improvements]
axes[1].bar(words_list, improvements, color=colors_imp, alpha=0.8, edgecolor='black')
axes[1].axhline(y=0, color='black', linewidth=1)
axes[1].set_ylabel('BiRNN Improvement over Forward RNN (%)', fontsize=11)
axes[1].set_title('How Much Better is BiRNN?\n(% improvement in disambiguation score)',
                  fontsize=12, fontweight='bold')
axes[1].set_xticklabels([f'"{w}"' for w in words_list], fontsize=12, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

for i, imp in enumerate(improvements):
    axes[1].text(i, imp + (1 if imp >= 0 else -3), f'+{imp:.0f}%' if imp >= 0 else f'{imp:.0f}%',
                 ha='center', fontsize=11, fontweight='bold',
                 color='#27AE60' if imp >= 0 else '#E74C3C')

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  Higher distance = the two sentence meanings produce MORE different")
print("  hidden states at the ambiguous word -- easier to disambiguate!")
print()
print("  The BiRNN's BACKWARD pass has already seen future words like")
print("  'river', 'cave', 'winter' when it processes the ambiguous word.")
print("  This extra context makes the representations more distinctive.")

---

## Part 7: Summary

### When to Use What

| Architecture | Best For | Key Trait |
|-------------|----------|------------|
| **Unidirectional RNN** | Text generation, streaming data, any task where the future is unknown | Only uses past context |
| **Bidirectional RNN** | Classification, NER, sentiment analysis, any task with the full sequence available | Uses past AND future context |
| **Stacked (Deep) RNN** | Complex tasks that need more capacity and abstraction | Multiple layers learn hierarchical features |
| **Stacked Bidirectional** | State-of-the-art sequence understanding (e.g., BERT-style encoders) | Best of both worlds, but most expensive |

### Key Takeaways

1. **Forward-only RNNs are limited** -- they cannot use future context to understand the present.

2. **Bidirectional RNNs** run two separate RNNs (forward + backward) and concatenate their outputs. This gives each position context from **both** directions.

3. **Output size doubles**: if hidden_size = $H$, the bidirectional output is $2H$.

4. **Only use bidirectional when the full sequence is available** before you need predictions. Do NOT use it for generative tasks.

5. **Stacking layers** lets the network learn more abstract features, with each layer building on the one below it.

6. In practice, frameworks like PyTorch make bidirectional and stacked RNNs a one-line configuration change: `nn.RNN(bidirectional=True, num_layers=2)`.

---

## Test Your Understanding

1. **Why can't a forward-only RNN disambiguate the word "bank" in "The bank by the river was muddy"?**
   <details>
   <summary>Click to reveal answer</summary>
   When the forward RNN processes "bank" (position 1), it has only seen "The" so far. The disambiguating word "river" appears later in the sentence, and a forward-only RNN has no way to access future tokens.
   </details>

2. **If a Bidirectional RNN has a hidden size of 128 per direction, what is the output dimension at each time step?**
   <details>
   <summary>Click to reveal answer</summary>
   256. The forward hidden state (128) and backward hidden state (128) are concatenated: 128 + 128 = 256.
   </details>

3. **Can you use a Bidirectional RNN for text generation (predicting the next word)?**
   <details>
   <summary>Click to reveal answer</summary>
   No. Text generation is an autoregressive task where you predict the next token one at a time. You cannot use a backward RNN because the future tokens do not exist yet. Use a unidirectional RNN (or Transformer decoder) instead.
   </details>

4. **In a 3-layer stacked BiRNN with hidden_size=64, what is the input size to Layer 2?**
   <details>
   <summary>Click to reveal answer</summary>
   128. Layer 1's output is the concatenation of forward (64) and backward (64) states, giving 128 dimensions. That becomes Layer 2's input.
   </details>

5. **Name two tasks where a Bidirectional RNN is a good choice, and two where it is NOT.**
   <details>
   <summary>Click to reveal answer</summary>
   Good: text classification, named entity recognition, sentiment analysis, machine translation (encoder side). Bad: text generation, real-time speech recognition, chatbot response generation, live stock prediction.
   </details>

---

## Interview Prep

Common interview questions about bidirectional and stacked RNNs. Practice explaining these out loud -- interviewers love candidates who can explain complex ideas simply.

**Q: When should you use a bidirectional RNN instead of a unidirectional one?**

Use bidirectional when you have the entire input sequence available before you need to produce output. Tasks like text classification, named entity recognition, and sentiment analysis are good fits. If you are generating output token by token (text generation, chatbots), stick with unidirectional because future tokens do not exist yet.

**Q: Why can't you use bidirectional RNNs for text generation?**

Text generation is autoregressive -- you predict one token at a time, and each prediction depends on the tokens generated so far. The backward RNN would need to read tokens that have not been produced yet, which is impossible. You must use a unidirectional (left-to-right) model for generation tasks.

**Q: What is the output dimension of a bidirectional RNN?**

If each direction has a hidden size of H, the output at each time step is 2H. That is because the forward hidden state (size H) and backward hidden state (size H) are concatenated. So a BiRNN with hidden_size=256 produces a 512-dimensional output at every position.

**Q: What are stacked/deep RNNs and when do you use them?**

A stacked RNN has multiple RNN layers where the output of one layer becomes the input to the next. Lower layers learn basic patterns (word-level features), and higher layers learn more abstract patterns (phrase-level or sentence-level features). You use them when your task is complex enough that a single layer does not have enough representational capacity. Two to four layers is common in practice; more than that often hits diminishing returns.

**Q: How does bidirectional processing help with word sense disambiguation?**

Many words have multiple meanings (e.g., "bank" can mean a river bank or a financial institution). A forward-only RNN at the word "bank" might have only seen "The" -- not enough to pick the right meaning. The backward RNN, however, has already seen the rest of the sentence (like "river" or "statement"), providing the disambiguating context. Concatenating both directions gives the model evidence from both sides of the ambiguous word.

**Q: What is the computational cost of bidirectional vs unidirectional?**

A bidirectional RNN roughly doubles the computation and parameters compared to a unidirectional RNN of the same hidden size, because you are running two separate RNNs. It also doubles memory usage since you store hidden states for both directions. In addition, you cannot start producing output until the entire input has been read, which adds latency for real-time applications.

**Q: Name a real-world model that uses bidirectional RNNs.**

ELMo (Embeddings from Language Models) uses a deep bidirectional LSTM to produce contextualized word embeddings. BERT also draws on the idea of bidirectional context, though it uses Transformers instead of RNNs. On the RNN side, many production NER systems and the encoder in early neural machine translation models (such as the original seq2seq with attention by Bahdanau et al.) use bidirectional LSTMs or GRUs.

---

## What's Next?

Nice work. You now understand how Bidirectional RNNs capture context from both directions and when to use them.

But even bidirectional RNNs have a limitation: for very long sequences, the hidden state still struggles to carry information across many time steps (the vanishing gradient problem). In the next notebook, we will explore a revolutionary idea that changed everything.

**Sequence-to-Sequence models with Attention** -- instead of cramming everything into a fixed-size hidden state, the model learns to look back at the most relevant parts of the input at each decoding step.

You will learn:
- The encoder-decoder architecture
- Why attention was a breakthrough
- How to build attention from scratch
- Visualizing attention weights

**Next up** --> [Notebook 07: Sequence-to-Sequence with Attention](07_seq2seq_attention.ipynb)

---

*You have finished Notebook 06. You are building a strong foundation in sequence modeling.*

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)